In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time, psutil, threading
import numpy as np
import matplotlib.pyplot as plt


In [2]:
# Literature values: Horowitz 2014, Eyeriss 2017
E   = {"L1": 5,   "L2": 20,  "L3": 100, "DRAM": 640}  # pJ per byte
LAT = {"L1": 1,   "L2": 4,   "L3": 15,  "DRAM": 100}  # ns per byte

L3_SIZE = 16 * 1024 * 1024  # 16 MB — run: lscpu | grep cache  to verify yours

def cache_level(weight_bytes):
    if   weight_bytes <=  64_000: return "L1"
    elif weight_bytes <= 512_000: return "L2"
    elif weight_bytes <= L3_SIZE: return "L3"
    else:                         return "DRAM"

def energy_nJ(n_params, act_bytes, n_samples, batch):
    """
    KEY FIX: per-sample cost = activation bytes only (NOT weight bytes).
    Weights are loaded ONCE if they fit in cache.
    """
    fp32 = n_params * 4
    int8 = n_params * 1
    nb   = max(1, n_samples // batch)

    naive = nb * fp32 * E["DRAM"] + n_samples * act_bytes * E["L1"]

    wl = cache_level(fp32)
    ws = (nb * fp32 * E["DRAM"] + n_samples * act_bytes * E["L1"]  # spills to DRAM
          if wl == "DRAM" else
          fp32 * E["DRAM"] + n_samples * act_bytes * E[wl])         # fits in cache

    wl8 = cache_level(int8)
    q   = (nb * int8 * E["DRAM"] + n_samples * act_bytes * E["L1"]
           if wl8 == "DRAM" else
           int8 * E["DRAM"] + n_samples * act_bytes * E[wl8])

    return naive/1e3, ws/1e3, q/1e3  # pJ → nJ


In [3]:
class PowerMonitor:
    """Samples CPU freq + utilization every 10ms. P ∝ f² × util (CMOS law)."""
    def __init__(self, interval=0.01):
        self.interval = interval
        self.samples  = []
        self._on      = False

    def start(self):
        self.samples = []; self._on = True
        self._t = threading.Thread(target=self._run, daemon=True)
        self._t.start()

    def stop(self):
        self._on = False; self._t.join()

    def _run(self):
        while self._on:
            f = psutil.cpu_freq().current
            u = psutil.cpu_percent(interval=None)
            self.samples.append((f / 2000)**2 * (u / 100))
            time.sleep(self.interval)

    def avg(self): return np.mean(self.samples) if self.samples else 0


In [4]:
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.c1, self.c2   = nn.Conv2d(ch,ch,3,padding=1), nn.Conv2d(ch,ch,3,padding=1)
        self.bn1, self.bn2 = nn.BatchNorm2d(ch), nn.BatchNorm2d(ch)
    def forward(self, x):
        return F.relu(self.bn2(self.c2(F.relu(self.bn1(self.c1(x))))) + x)

class MiniResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem   = nn.Conv2d(1, 32, 3, padding=1)      # 32 channels instead of 64
        self.blocks = nn.Sequential(*[ResBlock(32) for _ in range(3)])  # 3 blocks instead of 8
        self.head   = nn.Linear(32, 10)
    def forward(self, x):
        x = F.relu(self.stem(x))
        x = self.blocks(x).mean(dim=[2,3])
        return self.head(x)


In [6]:
# ── Cell 6: Model + Fast Training (combined) ──────────────────
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.c1, self.c2   = nn.Conv2d(ch,ch,3,padding=1), nn.Conv2d(ch,ch,3,padding=1)
        self.bn1, self.bn2 = nn.BatchNorm2d(ch), nn.BatchNorm2d(ch)
    def forward(self, x):
        return F.relu(self.bn2(self.c2(F.relu(self.bn1(self.c1(x))))) + x)

class MiniResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem   = nn.Conv2d(1, 32, 3, padding=1)
        self.blocks = nn.Sequential(*[ResBlock(32) for _ in range(3)])
        self.head   = nn.Linear(32, 10)
    def forward(self, x):
        x = F.relu(self.stem(x))
        x = self.blocks(x).mean(dim=[2,3])
        return self.head(x)

model    = MiniResNet()
n_params = sum(p.numel() for p in model.parameters())
print(f"Params: {n_params:,} | FP32: {n_params*4/1e6:.2f} MB")

# ── Fast Training ──────────────────────────────────────────────
transform    = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,),(0.3081,))])
train_ds     = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_ds      = datasets.MNIST('./data', train=False, download=True, transform=transform)

subset       = torch.utils.data.Subset(train_ds, range(10_000))
train_loader = DataLoader(subset, batch_size=512, shuffle=True)

opt = torch.optim.Adam(model.parameters(), 1e-3)
model.train()
for xb, yb in train_loader:
    opt.zero_grad(); F.cross_entropy(model(xb), yb).backward(); opt.step()
print("Training done (~10s)")


Params: 56,522 | FP32: 0.23 MB
Training done (~10s)


In [9]:
N = len(test_ds); ACT_B = 28*28*4; monitor = PowerMonitor()
ws_loader    = DataLoader(test_ds, batch_size=1000, shuffle=False)
naive_loader = DataLoader(test_ds, batch_size=64,   shuffle=False)

# ── NAIVE ──────────────────────────────────────────────
monitor.start(); t0 = time.perf_counter()
model.eval()
with torch.no_grad():
    for xb, _ in naive_loader:
        tmp = MiniResNet()
        tmp.load_state_dict({k:v.clone() for k,v in model.state_dict().items()})
        tmp.eval(); tmp(xb)
t_naive, p_naive = time.perf_counter()-t0, monitor.avg(); monitor.stop()

# ── WS FP32 ────────────────────────────────────────────
model.eval()
with torch.no_grad():                          # warm-up
    for xb,_ in ws_loader: model(xb); break

monitor.start(); t0 = time.perf_counter(); correct_ws = 0
with torch.no_grad():
    for xb, yb in ws_loader:
        correct_ws += (model(xb).argmax(1)==yb).sum().item()
t_ws, p_ws = time.perf_counter()-t0, monitor.avg(); monitor.stop()

# ── WS INT8 ────────────────────────────────────────────
qmodel = torch.quantization.quantize_dynamic(model, {nn.Linear, nn.Conv2d}, torch.qint8)
qmodel.eval()
with torch.no_grad():
    for xb,_ in ws_loader: qmodel(xb); break  # warm-up

monitor.start(); t0 = time.perf_counter(); correct_q = 0
with torch.no_grad():
    for xb, yb in ws_loader:
        correct_q += (qmodel(xb).argmax(1)==yb).sum().item()
t_q, p_q = time.perf_counter()-t0, monitor.avg(); monitor.stop()

e_naive, e_ws, e_q = energy_nJ(n_params, ACT_B, N, 64)

print(f"\n{'='*68}")
print(f"{'Strategy':<28} {'Time(s)':>7} {'CPU Pwr':>8} {'Mem E(nJ)':>12} {'Accuracy':>9}")
print(f"{'='*68}")
print(f"{'Naive (FP32, cold, b=64)':<28} {t_naive:>7.2f} {p_naive:>8.3f} {e_naive:>12.0f}   N/A")
print(f"{'WS FP32  (warm, b=1000)':<28} {t_ws:>7.2f} {p_ws:>8.3f} {e_ws:>12.0f} {correct_ws/N*100:>8.2f}%")
print(f"{'WS INT8  (quant, b=1000)':<28} {t_q:>7.2f} {p_q:>8.3f} {e_q:>12.0f} {correct_q/N*100:>8.2f}%")
print(f"Energy saving WS:    {e_naive/e_ws:.1f}×  |  INT8: {e_naive/e_q:.1f}×")
print(f"Speedup      WS:     {t_naive/t_ws:.1f}×  |  INT8: {t_naive/t_q:.1f}×")



Strategy                     Time(s)  CPU Pwr    Mem E(nJ)  Accuracy
Naive (FP32, cold, b=64)        4.54    0.527     22729426   N/A
WS FP32  (warm, b=1000)         3.99    0.797       771896    91.90%
WS INT8  (quant, b=1000)        4.14    0.733       192974    91.91%
Energy saving WS:    29.4×  |  INT8: 117.8×
Speedup      WS:     1.1×  |  INT8: 1.1×


In [8]:
# Replace the training loop section only
opt = torch.optim.Adam(model.parameters(), 1e-3)
model.train()

# 5 epochs on 10k samples — ~45 seconds
for epoch in range(5):
    total_loss = 0
    for xb, yb in train_loader:
        opt.zero_grad()
        loss = F.cross_entropy(model(xb), yb)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/5 | Loss: {total_loss/len(train_loader):.3f}")

# Quick accuracy check before inference
model.eval()
correct = 0
test_loader_quick = DataLoader(test_ds, batch_size=1000, shuffle=False)
with torch.no_grad():
    for xb, yb in test_loader_quick:
        correct += (model(xb).argmax(1) == yb).sum().item()
print(f"Accuracy after training: {correct/len(test_ds)*100:.2f}%")



Epoch 1/5 | Loss: 1.635
Epoch 2/5 | Loss: 1.257
Epoch 3/5 | Loss: 0.980
Epoch 4/5 | Loss: 0.769
Epoch 5/5 | Loss: 0.610
Accuracy after training: 91.90%
